# 06 - Pellet scoring validation (manual vs algorithmic)

**Purpose**: measure how well the algorithmic pellet-outcome classifier
agrees with manual scoring, as a prerequisite for trusting the automated
outcomes in the main analysis.

**Restricted to pillar trays** because the algorithm is designed for
that tray type only.

**Input**: ``data.manual_pelletdf`` and ``data.kinematicsdf``.


In [ ]:
# parameters
FIGSIZE_CM = (14, 5)
FIGSIZE_PER_PHASE = (9, 5)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, confusion_matrix

from endpoint_ck_analysis.config import EXAMPLE_OUTPUT_DIR
from endpoint_ck_analysis.data_loader import load_all
from endpoint_ck_analysis.helpers.plotting import stamp_version

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data = load_all()


## 1. Build the per-pellet validation dataframe

Inner-join manual and algorithmic scores on shared per-pellet keys.


In [ ]:
manual_pellet_pillar = data.manual_pelletdf[data.manual_pelletdf['tray_type'] == 'P']
kinematics_pillar = data.kinematicsdf[data.kinematicsdf['tray_type'] == 'P']

algo_per_segment = kinematics_pillar[
    ['subject_id', 'session_date', 'tray_type', 'run_number', 'segment_num', 'segment_outcome']
].drop_duplicates().rename(columns={'run_number': 'tray_number', 'segment_num': 'pellet_number'})

validation = manual_pellet_pillar[
    ['subject_id', 'session_date', 'tray_type', 'tray_number', 'pellet_number', 'score']
].merge(
    algo_per_segment,
    on=['subject_id', 'session_date', 'tray_type', 'tray_number', 'pellet_number'],
    how='inner',
)
print(f'Matched {len(validation)} pillar pellets between manual and algorithmic scoring')

manual_cat_map = {0: 'missed', 1: 'displaced', 2: 'retrieved'}
algo_cat_map = {
    'untouched': 'missed', 'uncertain': 'missed',
    'displaced_sa': 'displaced', 'displaced_outside': 'displaced',
    'retrieved': 'retrieved',
}
validation['manual_cat'] = validation['score'].map(manual_cat_map)
validation['algo_cat'] = validation['segment_outcome'].map(algo_cat_map)
validation['manual_contacted'] = validation['manual_cat'] != 'missed'
validation['algo_contacted'] = validation['algo_cat'] != 'missed'


## 2. Summary statistics


In [ ]:
three_way = (validation['manual_cat'] == validation['algo_cat']).mean()
binary = (validation['manual_contacted'] == validation['algo_contacted']).mean()
kappa = cohen_kappa_score(validation['manual_cat'], validation['algo_cat'])
print(f'Three-way exact agreement:     {three_way:.3%}')
print(f'Binary (contacted vs missed):  {binary:.3%}')
print(f"Cohen's kappa (three-way):     {kappa:.3f}")
print('Interpretation of kappa: <0.4 poor, 0.4-0.6 moderate, 0.6-0.8 substantial, >0.8 almost perfect')
print('\nConfusion matrix (rows=manual, cols=algorithmic):')
print(pd.crosstab(validation['manual_cat'], validation['algo_cat'], margins=True))


## 3. Confusion matrix heatmap (counts + row-normalized)


In [ ]:
cats = ['missed', 'displaced', 'retrieved']
cm = confusion_matrix(validation['manual_cat'], validation['algo_cat'], labels=cats)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_CM)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cats, yticklabels=cats,
            ax=axes[0], cbar_kws={'label': 'count'})
axes[0].set_xlabel('Algorithmic classification')
axes[0].set_ylabel('Manual classification')
axes[0].set_title('Pillar confusion matrix (raw counts)')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=cats, yticklabels=cats,
            ax=axes[1], vmin=0, vmax=1, cbar_kws={'label': 'fraction'})
axes[1].set_xlabel('Algorithmic classification')
axes[1].set_ylabel('Manual classification')
axes[1].set_title('Pillar confusion matrix (row-normalized)')
plt.tight_layout()
stamp_version(fig, label='06 confusion')
plt.savefig(EXAMPLE_OUTPUT_DIR / '06_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Per-phase agreement

Does the algorithm's accuracy drift across the experimental phases?


In [ ]:
validation_with_phase = validation.merge(
    manual_pellet_pillar[['subject_id', 'session_date', 'tray_number', 'pellet_number', 'phase_group']],
    on=['subject_id', 'session_date', 'tray_number', 'pellet_number'],
    how='left',
)
per_phase = validation_with_phase.groupby('phase_group').apply(
    lambda g: pd.Series({
        'n': len(g),
        'three_way_agreement': (g['manual_cat'] == g['algo_cat']).mean(),
        'binary_agreement': (g['manual_contacted'] == g['algo_contacted']).mean(),
    })
).sort_values('n', ascending=False)
print(per_phase)

fig, ax = plt.subplots(figsize=FIGSIZE_PER_PHASE)
x = range(len(per_phase))
ax.bar([i - 0.2 for i in x], per_phase['three_way_agreement'], width=0.4,
       label='Three-way agreement', color='steelblue')
ax.bar([i + 0.2 for i in x], per_phase['binary_agreement'], width=0.4,
       label='Binary agreement', color='orange')
ax.set_xticks(list(x))
ax.set_xticklabels(per_phase.index, rotation=45, ha='right')
ax.set_ylabel('Agreement rate')
ax.set_ylim(0, 1)
ax.axhline(0.9, color='green', linestyle='--', linewidth=0.7, label='0.90 reference')
ax.legend()
ax.set_title('Manual vs algorithmic agreement by phase (pillar trays only)')
for i, (phase, row) in enumerate(per_phase.iterrows()):
    ax.text(i, 0.02, f"N={int(row['n'])}", ha='center', fontsize=8)
plt.tight_layout()
stamp_version(fig, label='06 per phase')
plt.savefig(EXAMPLE_OUTPUT_DIR / '06_agreement_by_phase.png', dpi=150, bbox_inches='tight')
plt.show()
